# 9c. Routage et Repli : choisir le bon modèle, au bon moment

Ce notebook poursuit [09_Production_Patterns.ipynb](09_Production_Patterns.ipynb), qui traitait des
patterns d'appel à une API donnée (retry, streaming, rate limiting). Ici, la question change
d'échelle : **quand on dispose de plusieurs modèles et de plusieurs fournisseurs, qui répond à
quelle question — et que fait-on quand l'un d'eux tombe ?**

Trois briques de production :

1. **Le routage par politique** — router chaque question vers un modèle selon son coût, la
   difficulté estimée de la question et un budget par requête ;
2. **La cascade petit-vers-grand** — tenter d'abord le modèle économique, n'escalader vers le
   modèle coûteux que quand la réponse semble douteuse ;
3. **Le repli fournisseur** — survivre à la panne d'un fournisseur en basculant sur le suivant.

> **Terrain réel.** La flotte fait tourner un proxy multi-fournisseurs (OpenRouter, qui expose son
> catalogue et ses prix via `/v1/models`) et un service d'inférence local (Ollama sur ce poste).
> Ce notebook **lit** ce que ce terrain expose — catalogue, prix, latences — et ne touche jamais à
> sa configuration, qui appartient à un autre workspace.

> **Clés API.** Les clés vivent dans `.env` (cf. `00-GenAI-Environment`), chargées par
`python-dotenv`. Aucune clé n'est jamais écrite ni affichée dans ce notebook.


## Vérification de l'environnement


In [1]:
import os, json, time, random, re, statistics
from pathlib import Path
from dotenv import load_dotenv

# Le .env de la série GenAI (deux niveaux au-dessus du dossier Texte/)
load_dotenv(Path("..") / ".env")

OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
print("OPENROUTER_API_KEY :", "présente" if OPENROUTER_API_KEY else "ABSENTE -- sections proxy limitées")
print("OPENAI_API_KEY    :", "présente" if OPENAI_API_KEY else "absente")

# Reproductibilité : la banque de questions est générée avec une graine fixe
random.seed(20260928)


OPENROUTER_API_KEY : présente
OPENAI_API_KEY    : présente


In [2]:
# Sondage du terrain, en LECTURE SEULE : on interroge les catalogues, on ne configure rien
import urllib.request

def probe_catalogue(nom, url):
    req = urllib.request.Request(url)
    t0 = time.perf_counter()
    try:
        with urllib.request.urlopen(req, timeout=15) as r:
            data = json.loads(r.read().decode("utf-8"))
        dt = (time.perf_counter() - t0) * 1000
        modeles = [m["id"] for m in data.get("data", [])]
        print(f"{nom:38s} HTTP 200  {len(modeles):4d} modèles  ({dt:.0f} ms)")
        return modeles
    except Exception as e:
        print(f"{nom:38s} INJOIGNABLE ({type(e).__name__})")
        return []

ollama_modeles = probe_catalogue("Inference locale (Ollama)", "http://127.0.0.1:11434/v1/models")
or_modeles = probe_catalogue("Proxy multi-fournisseurs (OpenRouter)",
                             "https://openrouter.ai/api/v1/models")


Inference locale (Ollama)              HTTP 200     2 modèles  (93 ms)


Proxy multi-fournisseurs (OpenRouter)  HTTP 200   458 modèles  (184 ms)


### Lecture du terrain

Deux fournisseurs **réels et mesurables** répondent :

- l'**inférence locale** expose peu de modèles mais a un coût marginal nul et une latence sans
  réseau ;
- le **proxy multi-fournisseurs** expose des centaines de modèles avec leurs prix par token —
  c'est cette donnée de catalogue qui rend un routage économique *calculable*, et non intuitif.

La suite du notebook construit toutes ses décisions sur ces deux catalogues lus à l'instant.


## 1. Le catalogue et ses prix : la matière première du routeur

Un routeur qui arbitre sur le coût a besoin des coûts. L'API `/v1/models` du proxy renvoie, pour
chaque modèle, son prix par token d'entrée et de sortie — le terrain publie lui-même la table de
décision. On y ajoute le modèle local, dont le prix marginal est nul.


In [3]:
req = urllib.request.Request("https://openrouter.ai/api/v1/models")
with urllib.request.urlopen(req, timeout=15) as r:
    catalogue = {m["id"]: m for m in json.loads(r.read().decode("utf-8"))["data"]}

def prix_par_million(m):
    """Prix USD par million de tokens (entrée, sortie) — 0.0 pour les modèles :free."""
    p = m.get("pricing", {})
    try:
        return float(p.get("prompt", "1")) * 1e6, float(p.get("completion", "1")) * 1e6
    except (TypeError, ValueError):
        return float("nan"), float("nan")

# Candidats retenus pour la démonstration — choisis dans le catalogue RÉEL sondé ci-dessus
CANDIDATS_OR = [
    "mistralai/mistral-nemo",           # payant, quasi gratuit
    "meta-llama/llama-3.1-8b-instruct", # payant, petit
    "google/gemma-4-31b-it:free",       # grand, gratuit
]

catalogue_prix = {}
print(f"{'modèle':36s} {'$in/M':>9s} {'$out/M':>9s} {'ctx':>8s}  fournisseur")
for mid in CANDIDATS_OR:
    m = catalogue[mid]
    pin, pout = prix_par_million(m)
    catalogue_prix[mid] = {"in": pin, "out": pout, "ctx": m.get("context_length", 0)}
    print(f"{mid:36s} {pin:9.3f} {pout:9.3f} {m.get('context_length', 0):8d}  openrouter")

# Le modèle local : gratuit, mais limité au matériel présent
LOCAL_ID = "qwen2.5:7b"
LOCAL_KEY = f"local/{LOCAL_ID}"
catalogue_prix[LOCAL_KEY] = {"in": 0.0, "out": 0.0, "ctx": 32768}
print(f"{LOCAL_KEY:36s} {0.0:9.3f} {0.0:9.3f} {32768:8d}  ollama (local)")


modèle                                   $in/M    $out/M      ctx  fournisseur


mistralai/mistral-nemo                   0.019     0.030   131072  openrouter
meta-llama/llama-3.1-8b-instruct         0.050     0.080   131072  openrouter
google/gemma-4-31b-it:free               0.000     0.000   262144  openrouter
local/qwen2.5:7b                         0.000     0.000    32768  ollama (local)


### Lecture du catalogue

Le catalogue couvre un facteur de prix important entre le local gratuit et les modèles payants,
mais la ligne décisive pour la suite est ailleurs : **un grand modèle gratuit existe dans le
catalogue**. Le routage n'est donc pas seulement local-contre-payant — le proxy expose une vraie
diversité, et chaque brique du notebook mesurera ce que cette diversité vaut en exactitude, en
latence et en dollars.

Les prix affichés sont ceux publiés par le terrain au moment de l'exécution ; OpenRouter les fait
varier, et le notebook relit le catalogue à chaque passage plutôt que de les coder en dur.


## 2. Cent questions à réponse vérifiable

Mesurer un routeur exige de **noter les réponses automatiquement**. On génère donc une banque de
questions arithmétiques en français — problèmes à une ou deux étapes, chaque question portant sa
réponse exacte calculée par le générateur lui-même. La note est une égalité numérique stricte :
aucun juge, aucune ambiguïté.

La graine est fixée : la même banque sert à tous les modèles du notebook, et deux exécutions du
notebook produisent les mêmes questions.


In [4]:
def question_addition(rnd):
    a, b = rnd.randint(120, 980), rnd.randint(120, 980)
    q = f"Un entrepôt reçoit {a} colis le matin, puis {b} colis l'après-midi. Combien de colis au total ?"
    return q, a + b

def question_soustraction(rnd):
    a, b = rnd.randint(400, 990), rnd.randint(100, 390)
    q = f"Un magasin possède {a} unités de stock et en vend {b} dans la journée. Combien reste-t-il d'unités ?"
    return q, a - b

def question_produit(rnd):
    a, b = rnd.randint(12, 96), rnd.randint(12, 96)
    q = f"Une imprimerie produit {a} affiches par série et fabrique {b} séries. Combien d'affiches au total ?"
    return q, a * b

def question_pourcentage(rnd):
    p = rnd.choice([10, 15, 20, 25, 30, 40, 50, 75])
    total = rnd.randrange(200, 1000, 4)
    q = f"Un service compte {total} dossiers et {p}% sont traités en priorité. Combien de dossiers prioritaires ?"
    return q, total * p // 100

def question_deux_etapes(rnd):
    a, b, c = rnd.randint(15, 80), rnd.randint(3, 12), rnd.randint(20, 90)
    q = (f"Une équipe produit {a} pièces par heure pendant {b} heures, puis {c} pièces de plus "
         f"en heures supplémentaires. Combien de pièces au total ?")
    return q, a * b + c

GENES = [question_addition, question_soustraction, question_produit,
         question_pourcentage, question_deux_etapes]
GENE_NOMS = [g.__name__.replace("question_", "") for g in GENES]

rnd = random.Random(20260928)   # graine fixée : banque identique à chaque exécution
BANQUE = []
for i in range(100):
    g = GENES[i % len(GENES)]   # répartition égale : 20 questions par type
    texte, reponse = g(rnd)
    BANQUE.append({"id": i, "type": GENE_NOMS[i % len(GENES)],
                   "question": texte, "reponse": reponse})

print(f"Banque : {len(BANQUE)} questions, {len(GENES)} types")
for q in BANQUE[:5] + BANQUE[-2:]:
    print(f"  [{q['id']:3d}] ({q['type']:14s}) {q['question']}  ->  {q['reponse']}")


Banque : 100 questions, 5 types
  [  0] (addition      ) Un entrepôt reçoit 823 colis le matin, puis 682 colis l'après-midi. Combien de colis au total ?  ->  1505
  [  1] (soustraction  ) Un magasin possède 686 unités de stock et en vend 116 dans la journée. Combien reste-t-il d'unités ?  ->  570
  [  2] (produit       ) Une imprimerie produit 39 affiches par série et fabrique 24 séries. Combien d'affiches au total ?  ->  936
  [  3] (pourcentage   ) Un service compte 560 dossiers et 75% sont traités en priorité. Combien de dossiers prioritaires ?  ->  420
  [  4] (deux_etapes   ) Une équipe produit 40 pièces par heure pendant 8 heures, puis 85 pièces de plus en heures supplémentaires. Combien de pièces au total ?  ->  405
  [ 98] (pourcentage   ) Un service compte 472 dossiers et 20% sont traités en priorité. Combien de dossiers prioritaires ?  ->  94
  [ 99] (deux_etapes   ) Une équipe produit 53 pièces par heure pendant 4 heures, puis 74 pièces de plus en heures supplémentaires. Com

### Lecture de la banque

Les types couvrent une échelle de difficulté grossière : l'addition est triviale, le problème à
deux étapes demande de composer deux opérations. Cette échelle est **la variable que le routeur
essaiera de capter** — et la source de ses erreurs : la difficulté estimée n'est pas toujours la
difficulté réelle.


## 3. Baselines : chaque modèle face à la banque

Un client unique parle aux deux fournisseurs : Ollama et OpenRouter exposent tous deux l'API
compatible OpenAI. Le même code d'appel sert donc partout — c'est ce standard qui rend le repli
fournisseur possible au section 6.

La réponse est extraite par expression régulière (dernier nombre du texte rendu), et la note
compare à la réponse exacte du générateur.


In [5]:
from openai import OpenAI

def client_local():
    return OpenAI(base_url="http://127.0.0.1:11434/v1", api_key="ollama")

def client_openrouter():
    assert OPENROUTER_API_KEY, "OPENROUTER_API_KEY absente du .env"
    return OpenAI(base_url="https://openrouter.ai/api/v1", api_key=OPENROUTER_API_KEY)

def extraire_nombre(texte):
    """Dernier entier du texte rendu (le modèle conclut en général par la valeur)."""
    nombres = re.findall(r"-?\d+", texte)
    return int(nombres[-1]) if nombres else None

SYSTEME = ("Tu es un assistant de calcul. Réponds en une seule phrase courte et termine par la "
           "valeur numérique exacte, sans unité.")

def poser(client, modele, question, max_retries=3, delai_min=0.0):
    """Un appel, avec retry simple sur erreur. Renvoie (réponse_extraite, texte, latence_s).

    ``delai_min`` espace les appels pour respecter le quota du fournisseur : le terrain gratuit
    du proxy est limité à 30 requêtes/minute sur gemma-4-31b (mesuré : 429 au-delà). La latence
    rendue est celle de l'appel, hors délai d'espacement."""
    for tentative in range(max_retries + 1):
        try:
            t0 = time.perf_counter()
            rep = client.chat.completions.create(
                model=modele,
                messages=[{"role": "system", "content": SYSTEME},
                          {"role": "user", "content": question}],
                temperature=0.0, max_tokens=200,
            )
            dt = time.perf_counter() - t0
            texte = rep.choices[0].message.content or ""
            if delai_min and dt < delai_min:
                time.sleep(delai_min - dt)   # espacement anti-quota, hors mesure de latence
            return extraire_nombre(texte), texte, dt
        except Exception as e:
            if tentative == max_retries:
                raise
            # 429 = quota fournisseur : attendre la fenêtre de quota avant de retenter
            if "429" in str(e) or "RateLimit" in type(e).__name__:
                time.sleep(20.0 * (tentative + 1))
            else:
                time.sleep(1.5 * (tentative + 1))


In [6]:
def evaluer_banque(client, modele, banque=BANQUE, delai_min=0.0):
    """Fait passer la banque à un modèle. Renvoie la liste des résultats par question."""
    resultats = []
    for q in banque:
        pred, texte, dt = poser(client, modele, q["question"], delai_min=delai_min)
        resultats.append({**q, "pred": pred, "correct": pred == q["reponse"],
                          "latence": dt, "texte": (texte or "")[:120]})
    return resultats

def bilan(resultats, nom):
    n = len(resultats)
    acc = 100.0 * sum(r["correct"] for r in resultats) / n
    lat = statistics.mean(r["latence"] for r in resultats)
    print(f"{nom:34s} exactitude {acc:5.1f}%  latence moy. {lat:5.2f}s  (n={n})")
    return {"nom": nom, "accuracy": acc, "latence": lat}


### Passage 1 : le modèle local, gratuit

Les 100 questions passent d'abord au modèle local. C'est la mesure de référence du coût zéro :
ce que le matériel présent offre sans déboursé un centime.


In [7]:
t_debut = time.perf_counter()
res_local = evaluer_banque(client_local(), LOCAL_ID)
duree_local = time.perf_counter() - t_debut
bil_local = bilan(res_local, LOCAL_KEY)
print(f"passe complète en {duree_local:.0f} s")
par_type = {}
for r in res_local:
    par_type.setdefault(r["type"], []).append(r["correct"])
for t, cors in sorted(par_type.items()):
    print(f"  {t:14s} {100.0 * sum(cors) / len(cors):5.1f}% correct")


local/qwen2.5:7b                   exactitude  65.0%  latence moy.  0.19s  (n=100)
passe complète en 20 s
  addition        95.0% correct
  deux_etapes      5.0% correct
  pourcentage     60.0% correct
  produit         65.0% correct
  soustraction   100.0% correct


### Passage 2 : un grand modèle du proxy, gratuit lui aussi

Le catalogue expose un grand modèle gratuit. Le passer sur la même banque donne la deuxième
référence : celle du coût zéro distant — gratuit en dollars, payé en latence réseau et en
dépendance.


In [8]:
GEMMA = "google/gemma-4-31b-it:free"
t_debut = time.perf_counter()
res_gemma = evaluer_banque(client_openrouter(), GEMMA, delai_min=2.5)
duree_gemma = time.perf_counter() - t_debut
bil_gemma = bilan(res_gemma, GEMMA)
print(f"passe complète en {duree_gemma:.0f} s")
par_type = {}
for r in res_gemma:
    par_type.setdefault(r["type"], []).append(r["correct"])
for t, cors in sorted(par_type.items()):
    print(f"  {t:14s} {100.0 * sum(cors) / len(cors):5.1f}% correct")


google/gemma-4-31b-it:free         exactitude  88.0%  latence moy.  1.34s  (n=100)
passe complète en 251 s
  addition       100.0% correct
  deux_etapes    100.0% correct
  pourcentage     40.0% correct
  produit        100.0% correct
  soustraction   100.0% correct


In [9]:
import pandas as pd

comparatif_types = pd.DataFrame({
    t: {
        LOCAL_KEY: 100.0 * sum(r["correct"] for r in res_local if r["type"] == t) / 20,
        GEMMA: 100.0 * sum(r["correct"] for r in res_gemma if r["type"] == t) / 20,
    } for t in GENE_NOMS
})
comparatif_types.index.name = "type de question"
comparatif_types.round(1)


,addition,soustraction,produit,pourcentage,deux_etapes
type de question,,,,,
local/qwen2.5:7b,95.0,100.0,65.0,60.0,5.0
google/gemma-4-31b-it:free,100.0,100.0,100.0,40.0,100.0


### Lecture des baselines

Les deux modèles affichent leurs forces sur des types différents — c'est exactement la matière
d'un routeur : **si aucun modèle ne domine partout, le choix par question a de la valeur**. La
différence de latence entre local et distant complète le tableau : le dollar n'est pas la seule
monnaie.

*Les chiffres affichés proviennent de l'exécution committée avec ce notebook — chaque valeur de
prose de la suite cite cette exécution et pas une autre.*


## 4. Routage par politique : coût × difficulté × budget

Le routeur le plus simple est une **règle écrite avant l'expérience** : estimer la difficulté de
la question, comparer au budget, choisir le modèle. Ici l'estimation lit le type de question
(donnée par le générateur) — en production, l'estimateur serait un classifieur ou un petit
modèle, et ses erreurs feraient partie du coût.

La politique retenue : les questions estimées faciles vont au local gratuit, les autres au grand
modèle du proxy ; un budget par requête plafonne ce que le proxy peut coûter.


In [10]:
def difficulte_estimee(q):
    """Score de difficulté 0-3, calculé SANS la réponse — ce que le routeur sait en production."""
    base = {"addition": 0, "soustraction": 1, "pourcentage": 1,
            "produit": 2, "deux_etapes": 3}
    return base.get(q["type"], 1)

class RouteurPolitique:
    """Route statique : facile -> local, difficile -> proxy. Budget plafonné en $/1k tokens."""
    def __init__(self, seuil_difficulte=2, budget_1k=0.01):
        self.seuil = seuil_difficulte
        self.budget = budget_1k

    def choisir(self, q):
        if difficulte_estimee(q) < self.seuil:
            return LOCAL_KEY, client_local(), LOCAL_ID
        # Vérifie que le modèle distant respecte le budget (prix in+out pour ~1k tokens)
        px = catalogue_prix[GEMMA]
        cout_1k = (px["in"] + px["out"]) * 1000 / 1e6
        if cout_1k > self.budget:
            return LOCAL_KEY, client_local(), LOCAL_ID
        return GEMMA, client_openrouter(), GEMMA

routeur = RouteurPolitique(seuil_difficulte=2, budget_1k=0.01)

# Où va chaque question ?
destinations = {}
for q in BANQUE:
    dest, _, _ = routeur.choisir(q)
    destinations.setdefault(dest, []).append(q["id"])
for dest, ids in destinations.items():
    print(f"{dest:38s} {len(ids):3d} questions")


local/qwen2.5:7b                        60 questions
google/gemma-4-31b-it:free              40 questions


In [11]:
def evaluer_routeur(routeur, banque=BANQUE):
    resultats = []
    for q in banque:
        dest, client, modele = routeur.choisir(q)
        pred, texte, dt = poser(client, modele, q["question"],
                                delai_min=2.5 if dest == GEMMA else 0.0)
        resultats.append({**q, "pred": pred, "correct": pred == q["reponse"],
                          "latence": dt, "modele": dest})
    return resultats

res_route = evaluer_routeur(routeur)
bil_route = bilan(res_route, "routeur politique")
utilisations = {}
for r in res_route:
    utilisations.setdefault(r["modele"], [0, 0])
    utilisations[r["modele"]][0] += 1
    utilisations[r["modele"]][1] += r["correct"]
for dest, (n, nc) in utilisations.items():
    print(f"  {dest:38s} {n:3d} questions, {100.0 * nc / n:5.1f}% correct")


routeur politique                  exactitude  91.0%  latence moy.  1.06s  (n=100)
  local/qwen2.5:7b                        60 questions,  85.0% correct
  google/gemma-4-31b-it:free              40 questions, 100.0% correct


### Lecture du routage

Le routeur est jugé sur deux chiffres qui se lisent **ensemble** : son exactitude globale, et la
répartition qu'il a produite. Un routeur peut battre les deux baselines s'il envoie chaque type au
modèle qui le réussit ; il peut aussi les perdre si son estimateur de difficulté se trompe — et
comme l'estimateur lit ici le type déclaré (information parfaite), la mesure borne par le haut ce
qu'un estimateur réel obtiendrait.


## 5. Cascade petit-vers-grand sur seuil de confiance

Le routage statique décide **avant** de voir la réponse. La cascade décide **après** : le modèle
économique répond d'abord, il déclare lui-même sa confiance, et une confiance sous le seuil
déclenche une seconde demande au modèle grand.

Le point délicat — et ce que cette section **mesure** — est la calibration : combien de réponses
fausses le petit modèle déclare-t-il avec assurance (faux succès) ? Combien de réponses justes
escalade-t-il inutilement ? Ces deux taux valident ou cassent le seuil.


In [12]:
SYSTEME_CONFIANCE = ("Tu es un assistant de calcul. Réponds sur une seule ligne, au format exact : "
                     "REPONSE: <valeur> CONFIANCE: <nombre entre 0 et 1>")

def poser_avec_confiance(client, modele, question):
    t0 = time.perf_counter()
    rep = client.chat.completions.create(
        model=modele,
        messages=[{"role": "system", "content": SYSTEME_CONFIANCE},
                  {"role": "user", "content": question}],
        temperature=0.0, max_tokens=200,
    )
    dt = time.perf_counter() - t0
    texte = (rep.choices[0].message.content or "").replace(" ", "")
    m_val = re.search(r"REPONSE:\"?(-?\d+)", texte)
    m_conf = re.search(r"CONFIANCE:\"?([01]?[.,]?\d*)", texte)
    val = int(m_val.group(1)) if m_val else None
    try:
        conf = float(m_conf.group(1).replace(",", ".")) if m_conf and m_conf.group(1) else 0.5
    except ValueError:
        conf = 0.5
    return val, min(max(conf, 0.0), 1.0), dt

# Phase de calibration : on mesure la confiance déclarée sur la banque, SANS escalader
calibration = []
for q in BANQUE:
    val, conf, dt = poser_avec_confiance(client_local(), LOCAL_ID, q["question"])
    calibration.append({**q, "pred": val, "correct": val == q["reponse"], "confiance": conf})

conf_justes = [c["confiance"] for c in calibration if c["correct"]]
conf_fausses = [c["confiance"] for c in calibration if not c["correct"]]
n_fausses = len(conf_fausses)
faux_succes = sum(1 for c in calibration if not c["correct"] and c["confiance"] >= 0.8)
print(f"Confiance moyenne — justes : {statistics.mean(conf_justes):.2f}  "
      f"fausses : {statistics.mean(conf_fausses) if conf_fausses else float('nan'):.2f}")
print(f"Faux succès (réponse fausse, confiance >= 0.8) : {faux_succes} / {n_fausses} réponses fausses")


Confiance moyenne — justes : 1.00  fausses : 1.00
Faux succès (réponse fausse, confiance >= 0.8) : 21 / 21 réponses fausses


In [13]:
df_cal = pd.DataFrame(calibration)
table_calibration = pd.crosstab(
    pd.cut(df_cal["confiance"], [0, .4, .6, .8, 1.0], include_lowest=True),
    df_cal["correct"],
    rownames=["confiance déclarée"], colnames=["réponse exacte"])
table_calibration


réponse exacte,False,True
confiance déclarée,,
"(0.8, 1.0]",21,79


### Lecture de la calibration

La table croisée dit tout : si la confiance déclarée était calibrée, la colonne `True` dominerait
en haut et la colonne `False` en bas. Les lignes du haut à droite sont les **faux succès** — les
réponses fausses qui ne déclencheront jamais d'escalade. C'est le plafond d'exactitude de la
cascade : aucun seuil ne récupère une réponse fausse déclarée à 0.95.


In [14]:
SEUIL_CASCADE = 0.6   # choisi depuis la table de calibration ci-dessus

def cascade(question):
    """Local d'abord ; escalade vers le grand modèle si confiance < seuil."""
    val, conf, dt1 = poser_avec_confiance(client_local(), LOCAL_ID, question)
    if conf >= SEUIL_CASCADE:
        return val, f"local(conf={conf:.2f})", dt1, False
    val2, texte2, dt2 = poser(client_openrouter(), GEMMA, question, delai_min=2.5)
    return val2, "cascade->gemma", dt1 + dt2, True

res_cascade = []
n_escalades = 0
for q in BANQUE:
    val, origine, dt, escalade = cascade(q["question"])
    n_escalades += escalade
    res_cascade.append({**q, "pred": val, "correct": val == q["reponse"],
                        "latence": dt, "escalade": escalade})

bil_cascade = bilan(res_cascade, f"cascade (seuil {SEUIL_CASCADE})")
print(f"escalades vers le grand modèle : {n_escalades}/{len(BANQUE)}")


cascade (seuil 0.6)                exactitude  79.0%  latence moy.  0.32s  (n=100)
escalades vers le grand modèle : 0/100


### Lecture de la cascade

Trois chiffres jugent la cascade : exactitude finale, taux d'escalade, latence. La comparer aux
baselines du section 3 donne le verdict honnête — si le petit modèle est mal calibré, la cascade
peut coûter plus que le grand modèle seul pour un résultat égal. **Ce n'est pas un échec de la
méthode, c'est la mesure que la calibration doit précéder le seuil.**


## 6. Repli fournisseur sur panne simulée

Dernière brique : le proxy tombe. On simule la panne en pointant le client vers une URL valide en
apparence mais morte (le port d'un service arrêté) — le code d'appel, lui, reste celui de
production. La chaîne de repli essaie le maillon mort, puis bascule sur le proxy réel, puis sur
l'inférence locale.

On mesure ce que le repli **sauve** (requêtes servies malgré la panne) et ce qu'il **coûte**
(latence des tentatives mortes avant bascule).


In [15]:
PROXY_MORT = "http://127.0.0.1:11435/v1"   # port fermé : la panne simulée

def poser_avec_repli(question, chaine):
    """Essaie chaque (client, modèle) de la chaîne jusqu'à une réponse. Renvoie (pred, essais)."""
    essais = []
    for client, modele in chaine:
        try:
            t0 = time.perf_counter()
            pred, texte, _ = poser(client, modele, question, max_retries=0,
                                     delai_min=2.5 if modele == GEMMA else 0.0)
            essais.append((modele, "ok", time.perf_counter() - t0))
            return pred, essais
        except Exception as e:
            essais.append((modele, f"échec: {type(e).__name__}", time.perf_counter() - t0))
            continue
    return None, essais

CHAINE = [
    (OpenAI(base_url=PROXY_MORT, api_key="simulation"), GEMMA),  # simule le proxy en panne
    (client_openrouter(), GEMMA),                                 # proxy réel
    (client_local(), LOCAL_ID),                                   # dernier recours local
]

res_repli = []
for q in BANQUE[:30]:   # échantillon : la démonstration suffit, le motif est répétitif
    pred, essais = poser_avec_repli(q["question"], CHAINE)
    res_repli.append({**q, "pred": pred, "correct": pred == q["reponse"],
                      "essais": essais, "n_essais": len(essais)})

servies = sum(1 for r in res_repli if r["pred"] is not None)
justes = sum(1 for r in res_repli if r["correct"])
essais_morts = [r["essais"][0][2] for r in res_repli if r["n_essais"] > 1]
print(f"requêtes servies malgré la panne du premier maillon : {servies}/{len(res_repli)}")
print(f"dont exactes : {justes}/{len(res_repli)}")
if essais_morts:
    print(f"surcoût de latence de la tentative morte : {statistics.mean(essais_morts):.2f} s en moyenne")


requêtes servies malgré la panne du premier maillon : 30/30
dont exactes : 27/30
surcoût de latence de la tentative morte : 7.45 s en moyenne


### Lecture du repli

Le premier maillon étant mort par construction, chaque requête paie une tentative morte puis est
servie par le maillon suivant : **disponibilité totale sur cet échantillon, payée en secondes**.
En production, c'est l'arbitrage exact entre SLA de disponibilité et latence p99 — et la raison
pour laquelle une sonde de santé remplace avantageusement la tentative morte (un circuit ouvert
évite même d'essayer).


## 7. Synthèse : ce que chaque stratégie achète

Les passes de ce notebook sont réunies ici. Chaque ligne est une stratégie **réellement
exécutée** sur la même banque de 100 questions ; les colonnes sont les trois monnaies du routage.


In [16]:
def cout_usd(n_questions, modele_key):
    px = catalogue_prix[modele_key]
    # estimation : ~60 tokens d'entrée, ~30 de sortie par question (messages courts du notebook)
    return n_questions * (60 * px["in"] + 30 * px["out"]) / 1e6

synthese = pd.DataFrame([
    {**bil_local,   "coût USD": cout_usd(len(res_local), LOCAL_KEY),   "stratégie": "local seul"},
    {**bil_gemma,   "coût USD": cout_usd(len(res_gemma), GEMMA),      "stratégie": "grand distant seul"},
    {**bil_route,   "coût USD": cout_usd(n_escalades_route := sum(1 for r in res_route if r["modele"] == GEMMA), GEMMA), "stratégie": "routage politique"},
    {**bil_cascade, "coût USD": cout_usd(n_escalades, GEMMA),         "stratégie": "cascade"},
])[["stratégie", "accuracy", "latence", "coût USD"]]
synthese.round(4)


,stratégie,accuracy,latence,coût USD
0,local seul,65.0,0.1948,0.0
1,grand distant seul,88.0,1.3418,0.0
2,routage politique,91.0,1.0572,0.0
3,cascade,79.0,0.3202,0.0


### Lecture de la synthèse

**Aucune stratégie ne domine sur les trois colonnes** — c'est la leçon : le routage est un
arbitrage, pas un classement. Le local achète le coût et souvent la latence ; le distant achète
l'exactitude sur les types durs ; la cascade n'achète quelque chose que si la calibration la
porte ; le repli n'achète rien en exactitude mais tout en disponibilité.

Limites honnêtes de la mesure :

- la banque est arithmétique — elle étalonne le routage, pas le raisonnement long ;
- les prix sont ceux publiés par le proxy au moment de l'exécution ;
- une seule exécution par stratégie est committée : les modèles à température 0 sont stabilisés,
  mais la latence réseau varie d'un passage à l'autre ;
- l'estimateur de difficulté du routeur lit le type déclaré — information parfaite qu'un
  système réel n'a pas.


## Exercices

### Exercice 1 : budget par requête durci

Le `RouteurPolitique` vérifie un budget pour ~1k tokens. Durcissez-le : le routeur doit refuser
tout modèle dont le coût estimé dépasse un budget exprimé en **dollars par requête**, estimé
depuis les `max_tokens` et la longueur constatée des questions. Documentez ce que le budget
durci change aux destinations.

### Exercice 2 : cascade à trois niveaux

Ajoutez un niveau intermédiaire (par ex. `mistralai/mistral-nemo`, payant quasi gratuit) entre le
local et le grand : local → intermédiaire → grand. Quelle exactitude pour quel taux d'escalade,
par rapport à la cascade à deux niveaux ?

### Exercice 3 : circuit breaker

Le repli du section 6 essaie le maillon mort à chaque requête. Implémentez un circuit breaker :
après N échecs consécutifs, le maillon est déclaré ouvert pendant une durée de refroidissement,
et les requêtes vont directement au suivant. Mesurez le gain de latence sur les 30 mêmes
questions.


In [17]:
# Exercice 1 -- budget par requête durci
# TODO étudiant : spécialiser RouteurPolitique en RouteurBudgetDure
# Indice : estimer tokens_entree ~ 60 + len(question)//4, tokens_sortie ~ max_tokens
# Étape 1 : ajouter un paramètre budget_par_requete (USD)
# Étape 2 : calculer le coût pire-cas par modèle du catalogue
# Étape 3 : aiguiller vers le premier modèle sous budget, du moins cher au plus cher
print("Exercice à compléter")
resultat_ex1 = None  # TODO étudiant : DataFrame des destinations sous budget durci


Exercice à compléter


In [18]:
# Exercice 2 -- cascade à trois niveaux
# TODO étudiant : généraliser cascade() en cascade_niveaux(question, niveaux, seuils)
# Indice : chaque niveau porte son propre seuil de confiance
# Étape 1 : définir la liste [(local, 0.6), (nemo, 0.75), (gemma, None)]
# Étape 2 : escalader tant que la confiance déclarée est sous le seuil du niveau
# Étape 3 : mesurer exactitude, escalades et latence, comparer à la cascade à deux niveaux
print("Exercice à compléter")
resultat_ex2 = None  # TODO étudiant : bilan de la cascade à trois niveaux


Exercice à compléter


In [19]:
# Exercice 3 -- circuit breaker
# TODO étudiant : implémenter la classe CircuitBreaker
# Indice : états fermé / ouvert / semi-ouvert, compteur d'échecs, horodatage d'ouverture
# Étape 1 : CircuitBreaker(seuil=3, refroidissement_s=30) avec méthode autoriser()
# Étape 2 : poser_avec_repli() consulte le breaker avant chaque maillon, enregistre succès/échec
# Étape 3 : rejouer les 30 questions et comparer la latence moyenne au section 6
print("Exercice à compléter")
resultat_ex3 = None  # TODO étudiant : latence moyenne avec et sans breaker


Exercice à compléter


## Conclusion

### Récapitulatif

| Brique | Ce qu'elle décide | Ce qu'elle coûte si mal réglée |
|---|---|---|
| Catalogue et prix | la matière première | décision sur des prix périmés |
| Banque vérifiable | la mesure | conclusions non falsifiables |
| Routage politique | **avant** la réponse | erreurs d'estimation de difficulté |
| Cascade | **après** la réponse | faux succès mal calibrés |
| Repli fournisseur | **pendant** la panne | tentative morte payée en latence |

### La suite

Le notebook suivant (09d) garde la même banque et s'attaque aux **caches** : cache sémantique et
son taux de faux succès mesuré, cache de préfixe et son point d'équilibre.

### Sources

- Catalogue et prix : `GET /v1/models` du proxy multi-fournisseurs (lectures de ce notebook)
- Terrain local : service d'inférence local, `GET /v1/models`


## Navigation

**Série GenAI Texte** | Suivant : [10 — LocalLlama](10_LocalLlama.ipynb) | [↑ Sommaire série](README.md)
